# 20 · WGFMU IV sweep · L0 干跑 (dryrun)

**目的**：不连机，纯 Python 验证 `PulseTrainBuilder` + `WgfmuIVSweepConfig` 装配正确，波形形状符合预期。

**通过标准**：
1. 11 个递减脉冲，rise/fall 平滑
2. measure 窗都在平顶里
3. assertion 全过
4. `_dryrun_out/iv_sweep_plan.json` 生成成功

**这一步跑通才能进 L1（连机）**。


In [ ]:
import sys
from pathlib import Path

# 让 notebook 在 git clone 后无需 pip install 也能跑
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from fefetlab.measurements.wgfmu.pulse_builder import (
    PulseSegment, PulseTrainBuilder, linear_voltage_segments
)
from fefetlab.measurements.wgfmu.iv_sweep import WgfmuIVSweepConfig

print("imports OK")
print("project root:", ROOT)


## 1. 构造 11 个递减脉冲 (p 沟道 FeFET IDVG, 0 -> -2V)


In [ ]:
segments = linear_voltage_segments(
    v_start=0.0,
    v_stop=-2.0,
    n_points=11,
    t_rise_s=1e-6,
    t_high_s=2e-6,
    t_fall_s=1e-6,
    t_base_s=2e-6,
    measure_points=20,
    measure_average_s=100e-9,
)
print(f"built {len(segments)} segments")
print(f"first: v_pulse={segments[0].v_pulse}, last: v_pulse={segments[-1].v_pulse}")


## 2. 编译为 PulseTrainPlan


In [ ]:
builder = PulseTrainBuilder(pattern_name="dryrun_iv_pattern", v_init=0.0, v_base=0.0)
plan = builder.build(segments)

print(f"pattern_name        : {plan.pattern_name}")
print(f"v_init / v_base     : {plan.v_init} / {plan.v_base}")
print(f"len(segments)       : {len(plan.segments)}")
print(f"len(vectors)        : {len(plan.vectors)}  (每 segment 4 段 = rise/high/fall/base)")
print(f"len(measure_events) : {len(plan.measure_events)}")
print(f"total_duration_s    : {plan.total_duration_s*1e6:.2f} us")


## 3. 画时间-电压波形 + 标 measure window


In [ ]:
t, v = plan.waveform_samples(dt_s=5e-8)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t*1e6, v, lw=1.2, color="#1f77b4")
ax.set_xlabel("time (us)")
ax.set_ylabel("voltage (V)")
ax.set_title("Dryrun IV sweep waveform · 11 pulses, 0 -> -2V")
ax.grid(alpha=0.3)

# 标 measure window (取 high 段的起止)
for seg in plan.segments:
    ax.axvspan(seg["t_high_start_s"]*1e6, seg["t_high_end_s"]*1e6,
               alpha=0.18, color="orange")

ax.text(0.02, 0.95, "orange band = measure window (high plateau)",
        transform=ax.transAxes, va="top", fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
plt.tight_layout()
plt.show()


## 4. assertion · 自动检查


In [ ]:
errors = []

# 4.1 segment 数
if len(plan.segments) != 11:
    errors.append(f"segments count mismatch: expect 11, got {len(plan.segments)}")

# 4.2 measure event 数 (应与有 measure 的 segment 数一致)
n_measured = sum(1 for s in plan.segments if s["measure_points"] > 0)
if len(plan.measure_events) != n_measured:
    errors.append(f"measure_events mismatch: events={len(plan.measure_events)}, measured_segs={n_measured}")

# 4.3 每个 measure_event 必须落在对应 segment 的 high 平顶内
for (ev_name, t_start, points, interval, average, mode), seg in zip(plan.measure_events, plan.segments):
    t_end_meas = t_start + (points - 1) * interval
    if t_start < seg["t_high_start_s"] - 1e-12:
        errors.append(f"{ev_name}: t_start ({t_start:.3e}) < high_start ({seg['t_high_start_s']:.3e})")
    if t_end_meas > seg["t_high_end_s"] + 1e-12:
        errors.append(f"{ev_name}: t_end ({t_end_meas:.3e}) > high_end ({seg['t_high_end_s']:.3e})")
    if average >= interval:
        errors.append(f"{ev_name}: average_s ({average:.3e}) >= interval ({interval:.3e})")

# 4.4 电压单调递减
v_pulses = [s["v_pulse"] for s in plan.segments]
if not all(v_pulses[i] >= v_pulses[i+1] for i in range(len(v_pulses)-1)):
    errors.append(f"v_pulse not monotonic decreasing: {v_pulses}")

if errors:
    print("FAIL")
    for e in errors:
        print(" -", e)
    raise AssertionError(f"{len(errors)} dryrun assertion failure(s)")
else:
    print(f"PASS · all {n_measured} measure events in plateau, {len(plan.segments)} segments OK")


## 5. dump plan.json (人工核对用)


In [ ]:
out_dir = ROOT / "notebooks" / "_dryrun_out"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "iv_sweep_plan.json"

out_path.write_text(json.dumps({
    "pattern_name": plan.pattern_name,
    "v_init": plan.v_init,
    "v_base": plan.v_base,
    "total_duration_s": plan.total_duration_s,
    "n_vectors": len(plan.vectors),
    "n_measure_events": len(plan.measure_events),
    "segments": plan.segments,
    "measure_events": [
        {"event": e[0], "t_start_s": e[1], "points": e[2],
         "interval_s": e[3], "average_s": e[4], "mode": e[5]}
        for e in plan.measure_events
    ],
}, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"dumped: {out_path}")
print(f"size: {out_path.stat().st_size} bytes")


## 6. 顺手验证 WgfmuIVSweepConfig 能正常装配


In [ ]:
cfg = WgfmuIVSweepConfig(
    label="R1A_L0_dryrun",
    chan_id=101,
    v_init=0.0,
    v_base=0.0,
    operation_mode="FASTIV",
    force_voltage_range="AUTO",
    measure_mode="CURRENT",
    measure_current_range="1MA",
    treat_warning_as_error=False,
    timeout_s=60.0,
    sequence_count=1,
    notes="L0 dryrun, 0 -> -2V IDVG p-channel",
)
from dataclasses import asdict
print(json.dumps(asdict(cfg), ensure_ascii=False, indent=2))


---

## 通过标准回顾

- [ ] 第 3 步图看到 11 个等高度递减脉冲 (0 -> -2V)，orange 窗都在平顶里
- [ ] 第 4 步显示 `PASS`
- [ ] 第 5 步 `_dryrun_out/iv_sweep_plan.json` 生成
- [ ] 第 6 步 cfg 打印正常

**全部 ✅ → 进 22 (wakeup dryrun)**
